# Limpeza de Dados (Data Cleaning)
Este notebook trata os datasets `cobranca_assessorias.csv` e `fluxo_pagamentos.xlsx`, padroniza campos e corrige inconsistencias antes de gerar os arquivos limpos usados pelo ETL e pelo banco.

In [1]:
import os
import pandas as pd

BASE_DIR = os.path.abspath('..')
cobranca_path = os.path.join(BASE_DIR, 'cobranca_assessorias.csv')
pagamentos_path = os.path.join(BASE_DIR, 'fluxo_pagamentos.xlsx')

contratos = pd.read_csv(cobranca_path, encoding='latin1')
pagamentos = pd.read_excel(pagamentos_path)

print(f'Contratos originais: {contratos.shape}')
print(f'Pagamentos originais: {pagamentos.shape}')

Contratos originais: (10000, 8)
Pagamentos originais: (100000, 9)


## Padronizacao de dimensoes

In [2]:
contratos['Nome_Assessoria'] = contratos['Nome_Assessoria'].str.strip()
nome_lower = contratos['Nome_Assessoria'].str.lower()
contratos.loc[nome_lower.str.contains('rtice', na=False), 'Nome_Assessoria'] = 'Vertice Asset e Cobranca'
contratos.loc[nome_lower.str.contains('nix', na=False), 'Nome_Assessoria'] = 'Fenix Recuperacao de Credito'
contratos.loc[nome_lower.str.contains('nexus', na=False), 'Nome_Assessoria'] = 'Nexus Mediacao Financeira'
contratos.loc[nome_lower.str.contains('acerta', na=False), 'Nome_Assessoria'] = 'Acerta Credito Integrado'

contratos['Regiao_Cliente'] = contratos['Regiao_Cliente'].str.strip()
regiao_lower = contratos['Regiao_Cliente'].str.lower()
contratos.loc[regiao_lower == 'nordeste', 'Regiao_Cliente'] = 'Nordeste'
contratos.loc[regiao_lower == 'sudeste', 'Regiao_Cliente'] = 'Sudeste'
contratos.loc[regiao_lower == 'sul', 'Regiao_Cliente'] = 'Sul'
contratos.loc[regiao_lower.str.contains('centro', na=False), 'Regiao_Cliente'] = 'Centro-Oeste'
contratos.loc[regiao_lower == 'norte', 'Regiao_Cliente'] = 'Norte'

print('Assessorias padronizadas:')
print(contratos['Nome_Assessoria'].value_counts().sort_index())
print('\nRegioes padronizadas:')
print(contratos['Regiao_Cliente'].value_counts().sort_index())

Assessorias padronizadas:
Nome_Assessoria
Acerta Credito Integrado        1926
Fenix Recuperacao de Credito    3034
Nexus Mediacao Financeira       2037
Vertice Asset e Cobranca        3003
Name: count, dtype: int64

Regioes padronizadas:
Regiao_Cliente
Centro-Oeste     779
Nordeste        3691
Norte            484
Sudeste         3529
Sul             1517
Name: count, dtype: int64


## Tipos, nulos e outliers

In [3]:
def parse_money(val):
    if pd.isna(val):
        return 0.0
    s = str(val).strip().replace('R$', '').strip()
    if ',' in s:
        s = s.replace('.', '').replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return 0.0


score_nulos = int(contratos['Score_Interno_Risco'].isna().sum())
contratos['Valor_Inadimplente_Inicial'] = contratos['Valor_Inadimplente_Inicial'].apply(parse_money)
contratos['Score_Interno_Risco'] = contratos['Score_Interno_Risco'].fillna(contratos['Score_Interno_Risco'].median())
contratos['Data_Envio_Assessoria'] = pd.to_datetime(contratos['Data_Envio_Assessoria'], errors='coerce')

dias_validos = contratos['Dias_Em_Atraso_Inicial'].between(0, 365)
dias_invalidos = int((~dias_validos).sum())
mediana_dias_validos = int(contratos.loc[dias_validos, 'Dias_Em_Atraso_Inicial'].median())
contratos.loc[~dias_validos, 'Dias_Em_Atraso_Inicial'] = mediana_dias_validos

pagamentos['Data_Vencimento'] = pd.to_datetime(pagamentos['Data_Vencimento'], errors='coerce')
pagamentos['Data_Pagamento'] = pd.to_datetime(pagamentos['Data_Pagamento'], errors='coerce')
pagamentos['Forma_Pagamento'] = pagamentos['Forma_Pagamento'].str.strip()
pagamentos['Indicador_Contemplado'] = pagamentos['Indicador_Contemplado'].str.strip()

duplicatas_pagamentos = int(pagamentos.duplicated(subset=['ID_Pagamento']).sum())
pagamentos = pagamentos.drop_duplicates(subset=['ID_Pagamento'])

print(f'Scores nulos imputados pela mediana: {score_nulos}')
print(f'Atrasos invalidos tratados: {dias_invalidos}')
print(f'Mediana usada para atraso invalido: {mediana_dias_validos} dias')
print(f'Duplicatas de pagamento removidas: {duplicatas_pagamentos}')
print('\nAtraso inicial apos limpeza:')
print(contratos['Dias_Em_Atraso_Inicial'].describe())

Scores nulos imputados pela mediana: 300
Atrasos invalidos tratados: 80
Mediana usada para atraso invalido: 89 dias
Duplicatas de pagamento removidas: 0

Atraso inicial apos limpeza:
count    10000.000000
mean        89.645700
std         29.141728
min         15.000000
25%         70.000000
50%         89.000000
75%        109.000000
max        207.000000
Name: Dias_Em_Atraso_Inicial, dtype: float64


## Exportacao dos dados limpos

In [4]:
contratos.to_csv('contratos_clean.csv', index=False)
pagamentos.to_csv('pagamentos_clean.csv', index=False)

print('contratos_clean.csv e pagamentos_clean.csv atualizados.')
print(f'Contratos limpos: {contratos.shape}')
print(f'Pagamentos limpos: {pagamentos.shape}')

contratos_clean.csv e pagamentos_clean.csv atualizados.
Contratos limpos: (10000, 8)
Pagamentos limpos: (100000, 9)
